In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
import datetime

In [2]:
f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")

#change this so it's done after stacking
f_cleaned['Date'] = pd.to_datetime(f_cleaned['Date'], format = '%Y-%m-%d')
m_cleaned['Date'] = pd.to_datetime(m_cleaned['Date'], format = '%Y-%m-%d')

stacked = pd.concat([f_cleaned, m_cleaned], ignore_index = True)
df_sorted = stacked.sort_values(['Name', 'Year', 'Date'])

C:\Users\bnpar\AppData\Local\Temp\ipykernel_26752\4066794522.py:1: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
C:\Users\bnpar\AppData\Local\Temp\ipykernel_26752\4066794522.py:2: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")


Time from fir

In [3]:
df_sorted['TimeCompeting'] = pd.to_datetime(df_sorted['Year'].astype(str) + '-12-31') - df_sorted.groupby('Name')['Date'].transform('min')
df_sorted['TimeCompeting'] = df_sorted['TimeCompeting'].dt.days




In [4]:
bomb_squat = (
    ((df_sorted['Squat1Kg'] <= 0) | df_sorted['Squat1Kg'].isna()) &
    ((df_sorted['Squat2Kg'] <= 0) | df_sorted['Squat2Kg'].isna()) &
    ((df_sorted['Squat3Kg'] <= 0) | df_sorted['Squat3Kg'].isna()) & 
    ((df_sorted['Best3SquatKg'] <= 0) | df_sorted['Best3SquatKg'].isna())
)

bomb_bench = (
    ((df_sorted['Bench1Kg'] <= 0) | df_sorted['Bench1Kg'].isna()) &
    ((df_sorted['Bench2Kg'] <= 0) | df_sorted['Bench2Kg'].isna()) &
    ((df_sorted['Bench3Kg'] <= 0) | df_sorted['Bench3Kg'].isna()) & 
    ((df_sorted['Best3BenchKg'] <= 0) | df_sorted['Best3BenchKg'].isna())
)

bomb_deadlift = (
    ((df_sorted['Deadlift1Kg'] <= 0) | df_sorted['Deadlift1Kg'].isna()) &
    ((df_sorted['Deadlift2Kg'] <= 0) | df_sorted['Deadlift2Kg'].isna()) &
    ((df_sorted['Deadlift3Kg'] <= 0) | df_sorted['Deadlift3Kg'].isna()) & 
    ((df_sorted['Best3DeadliftKg'] <= 0) | df_sorted['Best3DeadliftKg'].isna())
)


df_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of times a lifter has bombed out. <br>
Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [5]:
####FIS THIS

df_sorted['NumberOfBombOuts'] = df_sorted.groupby('Name')['BombOut'].cumsum()
df_sorted['MeetsSinceLastBombOut'] = df_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount()

#past capped number of bombouts it gets set to cap
df_sorted.loc[df_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT

#if lifter has never bombed out number of meets since last bombout is set to cap
df_sorted.loc[(df_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT



In [6]:
attempt_cols = ['Squat1Kg','Squat2Kg','Squat3Kg',
                   'Bench1Kg','Bench2Kg','Bench3Kg',
                   'Deadlift1Kg','Deadlift2Kg','Deadlift3Kg']

df_sorted['AttemptsMade'] = (df_sorted[attempt_cols]>0).sum(axis=1)

#handling the case where the lifter didnt bombout but it's recorded as 0 attempts because individual attempst werent filled in 
mask = (df_sorted['BombOut']==False)&(df_sorted['AttemptsMade']==0)

#excluding lifters who bombed out form calculation as we already know this lifter did not bomb out
mean_attempts = df_sorted.loc[(df_sorted['BombOut']==False) & (df_sorted['AttemptsMade']>0),'AttemptsMade'].mean().round() 

df_sorted.loc[mask, 'AttemptsMade'] = mean_attempts

In [7]:
df_sorted['LastMeetAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('last')
df_sorted['AverageAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('mean')
df_sorted['BestMeetOfYear'] = df_sorted.groupby(['Name', 'Year'])['TotalKg'].transform('max')


In [8]:
# df_sorted['MeetsSinceLastBombOut'] = df_sorted.groupby(['Name', 'Year'])['MeetsSinceLastBombOut'].transform('last')
# df_sorted['NumberOfBombOuts'] = df_sorted.groupby(['Name', 'Year'])['NumberOfBombOuts'].transform('last')
df_sorted[df_sorted['AttemptsMade'] ==0]
print(len(df_sorted))
print(len(df_sorted[df_sorted['AttemptsMade'] ==0])/len(df_sorted))

515411
0.010416929401972407


In [9]:
panel_data = df_sorted.groupby(['Name', 'Year']).tail(1).reset_index(drop = True)

cols =['Name','Year', 'WeightClassKg', 'Sex','Age', 'TimeCompeting', 'Goodlift', 'BestMeetOfYear',  'AverageAttemptsMade', 'LastMeetAttemptsMade', 'MeetsSinceLastBombOut', 'NumberOfBombOuts']
panel_data = panel_data[cols]
panel_data = panel_data.sort_values(['Name', 'Year'])
panel_data['CompetesNextYear'] = (
    (panel_data['Name'] == panel_data['Name'].shift(-1)) & 
    (panel_data['Year'] +1 == panel_data['Year'].shift(-1))
)

#Need to drop the entries with the year as 2025 because we cannot know whether someone competed in 2026 yet.
max_year = panel_data['Year'].max()
panel_data = panel_data[panel_data['Year'] < max_year]

### Encoding categorical features
#### Encoding weight class

Classes encoded as (weight class rank)/(number of classes) such that the heaviest weight class present in a given year for a given sex is 1 and the lightest weight class present in that year is 0.

In [10]:
# f = panel_data[panel_data['Sex']== 'F'].copy()
# m = panel_data[panel_data['Sex']== 'M'].copy()


# f_classes = f['WeightClassKg'].unique()
# f_classes = [int(w_cl[:-1]) +0.5 if w_cl.endswith('+') else int(w_cl) for w_cl in f_classes]
# f_classes = sorted(f_classes)
# f_code = list(enumerate(f_classes))
# f_max_rank = f_code[-1][0]
# f_code_dict = {w_cl:i/f_max_rank for i,w_cl in f_code}

# m_classes = m['WeightClassKg'].unique()
# m_classes = [int(w_cl[:-1]) + 0.5 if w_cl.endswith('+') else int(w_cl) for w_cl in m_classes]
# m_classes = sorted(m_classes)
# m_code = list(enumerate(m_classes))
# m_max_rank = m_code[-1][0]
# m_code_dict = {w_cl:i/m_max_rank for i,w_cl in m_code}

# code_dict = f_code_dict
# code_dict.update(m_code_dict)
# panel_data['WeightClassKgCode'] = (
#     panel_data['WeightClassKg']
#     .apply(lambda x: int(x[:-1]) if x.endswith('+') else int(x))
#     .map(code_dict)
#                                   )

#panel_data['EncWeightClassKg'] = panel_data['WeightClassKg'].map(f_classes)

In [11]:
def w_cl_to_num(w):
    w = str(w)
    return float(w[:-1]) + 0.5 if w.endswith('+') else float(w)

panel_data['WeightClassKgNum'] = panel_data['WeightClassKg'].apply(w_cl_to_num)

panel_data['WeightClassKgCode'] = (
    panel_data.groupby(['Sex', 'Year'])['WeightClassKgNum']
    .transform(lambda x: (x.rank(method='dense') - 1) / (x.nunique() - 1))
)

panel_data = panel_data.drop(columns=['WeightClassKgNum'])

In [12]:
###one  hot encoding used for Sex 

panel_data['F'] = panel_data['Sex'].apply(lambda x: 1 if x == 'F' else 0)
panel_data['M'] = panel_data['Sex'].apply(lambda x: 1 if x == 'M' else 0)
panel_data['Mx'] = panel_data['Sex'].apply(lambda x: 1 if x == 'Mx' else 0)

In [13]:
panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data.to_csv(panel_data_path, index = False)

In [14]:
###EDA vibes
null_attempt = df_sorted[
    df_sorted['Squat1Kg'].isna() |
    df_sorted['Squat2Kg'].isna() |
    df_sorted['Squat3Kg'].isna() |
    df_sorted['Bench1Kg'].isna() |
    df_sorted['Bench2Kg'].isna() |
    df_sorted['Bench3Kg'].isna() |
    df_sorted['Deadlift1Kg'].isna() |
    df_sorted['Deadlift2Kg'].isna() |
    df_sorted['Deadlift3Kg'].isna()
]

all_attempts_null = df_sorted[
    df_sorted['Squat1Kg'].isna() &
    df_sorted['Squat2Kg'].isna() &
    df_sorted['Squat3Kg'].isna() &
    df_sorted['Bench1Kg'].isna() &
    df_sorted['Bench2Kg'].isna() &
    df_sorted['Bench3Kg'].isna() &
    df_sorted['Deadlift1Kg'].isna() &
    df_sorted['Deadlift2Kg'].isna() &
    df_sorted['Deadlift3Kg'].isna()
]

print(len(null_attempt)/len(df_sorted))
print(len(all_attempts_null)/ len(df_sorted))

failed_attempt = df_sorted[
    (df_sorted['Squat1Kg'] < 0) |
    (df_sorted['Squat2Kg'] < 0) |
    (df_sorted['Squat3Kg'] < 0) |
    (df_sorted['Bench1Kg'] < 0) |
    (df_sorted['Bench2Kg'] < 0) |
    (df_sorted['Bench3Kg'] < 0) |
    (df_sorted['Deadlift1Kg'] < 0) |
    (df_sorted['Deadlift2Kg'] < 0) |
    (df_sorted['Deadlift3Kg'] < 0)
]


0.22394361005100782
0.17179299626899697
